# NASA C-MAPSS Remaining Useful Life (RUL) Prediction
### Baseline Model: Random Forest
This notebook loads the FD001 dataset, calculates the RUL, and trains a baseline model.

In [1]:
import pandas as pd
import numpy as np
import os
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error

### 1. Load Data

In [2]:
columns = ['unit_nr', 'time_cycles', 'setting_1', 'setting_2', 'setting_3'] + ['s_' + str(i) for i in range(1,22)]

train = pd.read_csv('data/train_FD001.txt', sep='\s+', header=None, names=columns)
test = pd.read_csv('data/test_FD001.txt', sep='\s+', header=None, names=columns)
y_test = pd.read_csv('data/RUL_FD001.txt', sep='\s+', header=None, names=['RUL'])

train.head()

,unit_nr,time_cycles,setting_1,setting_2,setting_3,s_1,s_2,s_3,s_4,s_5,...,s_12,s_13,s_14,s_15,s_16,s_17,s_18,s_19,s_20,s_21
0,1,1,-0.0007,-0.0004,100.0,518.67,641.82,1589.70,1400.60,14.62,...,521.66,2388.02,8138.62,8.4195,0.03,392,2388,100.0,39.06,23.4190
1,1,2,0.0019,-0.0003,100.0,518.67,642.15,1591.82,1403.14,14.62,...,522.28,2388.07,8131.49,8.4318,0.03,392,2388,100.0,39.00,23.4236
2,1,3,-0.0043,0.0003,100.0,518.67,642.35,1587.99,1404.20,14.62,...,522.42,2388.03,8133.23,8.4178,0.03,390,2388,100.0,38.95,23.3442
3,1,4,0.0007,0.0000,100.0,518.67,642.35,1582.79,1401.87,14.62,...,522.86,2388.08,8133.83,8.3682,0.03,392,2388,100.0,38.88,23.3739
4,1,5,-0.0019,-0.0002,100.0,518.67,642.37,1582.85,1406.22,14.62,...,522.19,2388.04,8133.80,8.4294,0.03,393,2388,100.0,38.90,23.4044


### 2. Calculate RUL for Training Data

In [3]:
def add_rul(df):
    max_cycle = df.groupby('unit_nr')['time_cycles'].max().reset_index()
    max_cycle.columns = ['unit_nr', 'max_cycle']
    df = df.merge(max_cycle, on=['unit_nr'], how='left')
    df['RUL'] = df['max_cycle'] - df['time_cycles']
    df.drop('max_cycle', axis=1, inplace=True)
    return df

train = add_rul(train)

### 3. Feature Selection & Model Training

In [4]:
drop_cols = ['unit_nr', 'time_cycles', 'setting_1', 'setting_2', 'setting_3', 
             's_1', 's_5', 's_6', 's_10', 's_16', 's_18', 's_19']
features = [c for c in train.columns if c not in drop_cols and c != 'RUL']

X_train = train[features]
y_train = train['RUL']

X_test = test.groupby('unit_nr').last().reset_index()[features]

rf = RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42)
rf.fit(X_train, y_train)

RandomForestRegressor(max_depth=10, random_state=42)

### 4. Evaluation

In [5]:
preds = rf.predict(X_test)
rmse = np.sqrt(mean_squared_error(y_test['RUL'], preds))
print(f"Baseline Random Forest RMSE: {rmse:.2f}")

importances = rf.feature_importances_
indices = np.argsort(importances)[::-1]
print("\nTop 5 Important Sensors:")
for i in range(5):
    print(f"Sensor {features[indices[i]].replace('s_', '')}: {importances[indices[i]]:.4f}")

Baseline Random Forest RMSE: 32.17

Top 5 Important Sensors:
Sensor 11: 0.5859
Sensor 9: 0.1438
Sensor 4: 0.0997
Sensor 12: 0.0347
Sensor 7: 0.0246
